# 🔋 DEARLIBS Battery Model Implementation in Python

This notebook implements the DEARLIBS (Doyle-Fuller-Newman based Electrochemical model with Analytical Reformulation) pseudo-code in Python using symbolic mathematics (`SymPy`), numerical solvers (`SciPy`), and optimization (`pyswarms`).

In [2]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sympy.utilities.lambdify import lambdify
from pyswarms.single import GlobalBestPSO


## 🔧 Step 1: Define Symbolic Variables and Model Parameters

In [3]:
# Define time and parameters
t = sp.symbols('t')
kk = sp.symbols('k1:9')  # 8 parameters to identify

# Example placeholder variables (more can be added later)
y1, y2, y3 = sp.Function('y1')(t), sp.Function('y2')(t), sp.Function('y3')(t)
z1 = sp.Function('z1')(t)

# Define example ODEs and algebraic equation (to be expanded with real model)
f1 = -kk[0]*y1 + kk[1]*y2
f2 = kk[2]*y1 - kk[3]*y3
g1 = y1 + y2 + y3 - 1  # Example algebraic constraint

# System of equations
odes = [sp.Eq(y1.diff(t), f1), sp.Eq(y2.diff(t), f2)]
aes = [sp.Eq(0, g1)]


## 🧮 Step 2: Convert Symbolic Equations to Mass Matrix Form

In [4]:
def mass_matrix_form(eqs, vars, t):
    n = len(vars)
    M = sp.zeros(n)
    f_vec = sp.zeros(n, 1)
    for i, eq in enumerate(eqs):
        deriv = sp.Derivative(vars[i], t)
        coeff = eq.lhs.coeff(deriv)
        M[i, i] = coeff if coeff != 0 else 0
        f_vec[i] = eq.rhs - coeff * deriv if coeff != 0 else eq.rhs
    return M, f_vec

# Define variable list for mass matrix extraction
varsX = [y1, y2]
M_sym, f_sym = mass_matrix_form(odes, varsX, t)

# Lambdify M and f for numerical use
variables = [t] + varsX + list(kk)
M_func = lambdify(variables, M_sym, modules='numpy')
f_func = lambdify(variables, f_sym, modules='numpy')


PrintMethodNotImplementedError: Unsupported by <class 'sympy.printing.numpy.NumPyPrinter'>: <class 'sympy.core.function.Derivative'>
Printer has no method: _print_Derivative_Dummy
Set the printer option 'strict' to False in order to generate partially printed code.

## ⚙️ Step 3: Define ODE Solver Wrapper

In [ ]:
def ode_system(t, y, params):
    args = [t] + list(y) + list(params)
    M = np.array(M_func(*args), dtype=float)
    f = np.array(f_func(*args), dtype=float).flatten()
    return np.linalg.solve(M, f)


## 🎯 Step 4: Define Objective Function for Optimization

In [ ]:
# Example synthetic voltage data
v_exp = np.linspace(3.0, 4.2, 50)

def compute_voltage(y):
    # Placeholder voltage from state vector
    return y[0, :]  # Simulated voltage

def objective(params):
    y0 = [0.9, 0.1]  # Example ICs
    t_span = (0, 100)
    try:
        sol = solve_ivp(lambda t, y: ode_system(t, y, params),
                        t_span=t_span, y0=y0, method='BDF',
                        rtol=1e-6, atol=1e-6)
        v_model = compute_voltage(sol.y)
        v_model_interp = np.interp(np.linspace(0, 100, len(v_exp)), sol.t, v_model)
        return np.mean((v_model_interp - v_exp) ** 2)
    except Exception as e:
        return 1e6  # Penalize failure


## 🐝 Step 5: Optimize Parameters Using Particle Swarm Optimization

In [ ]:
# Define bounds
p0 = np.ones(8)
lb = 0.7 * p0
ub = 1.3 * p0

optimizer = GlobalBestPSO(n_particles=10, dimensions=8, options={'c1': 0.5, 'c2': 0.3, 'w': 0.9}, bounds=(lb, ub))
best_cost, best_params = optimizer.optimize(objective, iters=10)

print("Best parameters:", best_params)


## 📉 Step 6: Final Simulation and Visualization

In [ ]:
sol = solve_ivp(lambda t, y: ode_system(t, y, best_params),
                t_span=(0, 100), y0=[0.9, 0.1], method='BDF')

v_sim = compute_voltage(sol.y)
v_interp = np.interp(np.linspace(0, 100, len(v_exp)), sol.t, v_sim)

plt.figure(figsize=(10, 5))
plt.plot(np.linspace(0, 100, len(v_exp)), v_exp, 'ro', label='Experimental Voltage')
plt.plot(np.linspace(0, 100, len(v_exp)), v_interp, 'b-', label='Simulated Voltage')
plt.xlabel("Time")
plt.ylabel("Voltage (V)")
plt.title("Battery Voltage Comparison")
plt.legend()
plt.grid(True)
plt.show()
